# 🔍 Simple RAG with Agent Framework

## 🎯 What You'll Learn

In this notebook, you'll learn how to build a **simple RAG (Retrieval-Augmented Generation)** system using Microsoft's Agent Framework. This is the foundation for building AI applications that can search through documents and provide intelligent answers.

### 🔍 **What is RAG?**
- **R**etrieval: Find relevant documents from a knowledge base
- **A**ugmented: Enhance the AI's knowledge with retrieved information  
- **G**eneration: Generate answers using both the AI's training and retrieved docs

### 🎯 **Why Agent Framework?**
- 🚀 **Simple Setup** - Built specifically for AI agent workflows
- 🔧 **Powerful Tools** - Pre-built components for common AI tasks
- 📊 **Built-in Observability** - Automatic tracing and monitoring
- 🔗 **Easy Integration** - Works seamlessly with Azure AI and GitHub Models

### 📚 **What We'll Build**
A simple RAG system that can:
- Store documents in a searchable knowledge base
- Find relevant documents based on user questions
- Generate intelligent answers using retrieved information

---

## 🛠️ Let's Get Started!

## 🔐 Prerequisites & Setup

### 📋 **Requirements**
- ✅ **GitHub Account** with Models access OR **Azure OpenAI** account
- ✅ **Python 3.10+** installed
- ✅ **Environment Variables** configured (see below)

### 🔐 **Environment Configuration**

**Option A: Using GitHub Models (Recommended for Learning)**
```env
GITHUB_TOKEN=your_github_personal_access_token
```

**Option B: Using Azure OpenAI**
```env
AZURE_OPENAI_API_KEY=your_api_key_here
AZURE_OPENAI_ENDPOINT=https://your-resource.openai.azure.com/
AZURE_OPENAI_DEPLOYMENT_NAME=your_gpt_deployment_name
AZURE_OPENAI_ADA_DEPLOYMENT=your_text_embedding_deployment_name
```

> **💡 Tip:** Create a `.env` file in your project root with these variables.

---

In [ ]:
# 📦 Install Required Packages
%pip install agent-framework python-dotenv numpy chromadb --quiet

print("✅ Agent Framework installed successfully!")
print("✅ ChromaDB vector database installed!")
print("📚 Ready to build your RAG system with embeddings!")

In [ ]:
# 🔧 Setup and Configuration
import os
import numpy as np
from dotenv import load_dotenv
from agent_framework.openai import OpenAIChatClient
from agent_framework.azure import AzureOpenAIChatClient
from agent_framework.observability import setup_observability
from azure.identity import DefaultAzureCredential

# Load environment variables
load_dotenv()

# 📊 Setup observability (optional but recommended)
try:
    setup_observability(
        otlp_endpoint="http://localhost:4317",  # AI Toolkit endpoint
        enable_sensitive_data=True  # Enable capturing prompts and completions
    )
    print("📊 Telemetry enabled for observability")
except:
    print("📊 Telemetry not available (optional)")

print("✅ Agent Framework configured")
print("🚀 Ready to build with vector embeddings!")

## 📚 Step 1: Create Our Knowledge Base

First, let's create some sample documents about AI and programming topics that our RAG system can search through.

In [ ]:
# 📄 Create Sample Documents
knowledge_base = [
    {
        "id": "doc_001",
        "title": "Introduction to AI Agents",
        "content": "AI agents are autonomous software entities that can perceive their environment, make decisions, and take actions to achieve specific goals. They use machine learning, natural language processing, and reasoning capabilities to interact with users and systems. Modern AI agents can handle complex workflows, integrate with multiple tools, and provide intelligent automation for various business processes.",
        "category": "AI Fundamentals"
    },
    {
        "id": "doc_002",
        "title": "What is RAG?",
        "content": "Retrieval-Augmented Generation (RAG) is a technique that enhances large language models by providing them with relevant external information during the generation process. RAG systems first retrieve relevant documents from a knowledge base, then use this context to generate more accurate and informed responses. This approach helps overcome the limitations of pre-trained models by incorporating up-to-date and domain-specific information.",
        "category": "RAG Technology"
    },
    {
        "id": "doc_003",
        "title": "Python for AI Development", 
        "content": "Python is the most popular programming language for AI development due to its simplicity, rich ecosystem of libraries, and strong community support. Key libraries include TensorFlow and PyTorch for deep learning, scikit-learn for machine learning, NumPy for numerical computing, and pandas for data manipulation. Python's readable syntax makes it ideal for rapid prototyping and experimentation in AI research.",
        "category": "Programming"
    },
    {
        "id": "doc_004",
        "title": "Vector Embeddings Explained",
        "content": "Vector embeddings are numerical representations of text, images, or other data that capture semantic meaning in a high-dimensional space. In AI applications, embeddings allow models to understand similarity and relationships between different pieces of content. They are created using neural networks trained to map similar concepts to nearby points in vector space, enabling efficient similarity search and retrieval.",
        "category": "AI Fundamentals"
    },
    {
        "id": "doc_005",
        "title": "Building Conversational AI",
        "content": "Conversational AI systems combine natural language understanding, dialogue management, and response generation to create human-like interactions. These systems use intent recognition to understand user goals, maintain context across conversation turns, and generate appropriate responses. Modern conversational AI leverages large language models, fine-tuning techniques, and integration with external systems to provide helpful and engaging user experiences.",
        "category": "AI Applications"
    }
]

print(f"✅ Knowledge base created with {len(knowledge_base)} documents")
print("\n📄 Available Documents:")
for doc in knowledge_base:
    print(f"   • {doc['title']} ({doc['category']})")

print("\n🔍 Ready for retrieval and generation!")

## 🧮 Step 1.5: Create Vector Embeddings

For true semantic search in RAG, we need to convert our documents into **vector embeddings**. This allows us to find documents based on meaning, not just keywords!

In [ ]:
# 🧮 Create Embeddings and Vector Store with ChromaDB
import chromadb
from chromadb.utils import embedding_functions

# Initialize ChromaDB client (in-memory for this demo)
chroma_client = chromadb.Client()

# Create or get the collection
# ChromaDB will automatically handle embeddings using sentence-transformers
try:
    # Try to get existing collection first
    collection = chroma_client.get_collection(name="ai_knowledge_base")
    print("📚 Using existing collection")
except:
    # Create new collection if it doesn't exist
    collection = chroma_client.create_collection(
        name="ai_knowledge_base",
        metadata={"description": "AI and programming knowledge base with semantic search"}
    )
    
    # Add documents to the collection
    # ChromaDB will automatically create embeddings for us!
    ids = [doc["id"] for doc in knowledge_base]
    documents = [doc["content"] for doc in knowledge_base]
    metadatas = [
        {
            "title": doc["title"],
            "category": doc["category"]
        } 
        for doc in knowledge_base
    ]
    
    collection.add(
        ids=ids,
        documents=documents,
        metadatas=metadatas
    )
    print("✅ Created new collection and added documents")

print(f"\n🧮 Vector Store Ready!")
print(f"   • Collection: {collection.name}")
print(f"   • Documents: {collection.count()}")
print(f"   • Embedding model: sentence-transformers (all-MiniLM-L6-v2)")
print(f"   • Each document is now a 384-dimensional vector")

# Let's peek at what an embedding looks like
sample_embedding = collection.get(
    ids=[knowledge_base[0]["id"]],
    include=["embeddings"]
)

if sample_embedding and sample_embedding.get("embeddings") is not None and len(sample_embedding["embeddings"]) > 0:
    embedding = sample_embedding["embeddings"][0]
    embedding_dim = len(embedding)
    print(f"\n📊 Embedding Details:")
    print(f"   • Dimensions: {embedding_dim}")
    print(f"   • Sample values: {[f'{x:.4f}' for x in embedding[:5]]}")
    print(f"\n💡 **What are embeddings?**")
    print(f"   Vector embeddings convert text into {embedding_dim}-dimensional numerical vectors")
    print(f"   that capture semantic meaning. Similar texts have similar vectors!")

print(f"\n🔍 Now ready for semantic similarity search!")

## 🤖 Step 2: Create Vector-Based RAG Agent

Now let's create a RAG agent that uses **vector similarity search** to find the most relevant documents based on semantic meaning!

In [ ]:
# 🤖 Create Vector-Based RAG Agent
from typing import Annotated
from pydantic import Field

class VectorRAGAgent:
    """A RAG agent that uses vector similarity search for semantic retrieval."""
    
    def __init__(self, vector_store):
        self.vector_store = vector_store
        
        # Setup chat client and create agent with vector search tool
        chat_client = self._setup_chat_client()
        self.agent = chat_client.create_agent(
            instructions=(
                "You are a helpful AI assistant with access to a knowledge base that uses semantic search. "
                "When users ask questions, use the search_documents tool to find relevant information. "
                "The search uses vector embeddings to find semantically similar content, not just keyword matches. "
                "Always provide comprehensive answers based on the retrieved documents and cite your sources."
            ),
            name="VectorRAGAgent",
            tools=[self._create_search_function()]
        )
    
    def _setup_chat_client(self):
        """Setup chat client with GitHub Models or Azure OpenAI."""
        try:
            # Try GitHub Models first (free and easy)
            if os.getenv("GITHUB_TOKEN"):
                return OpenAIChatClient(
                    model_id="gpt-4o-mini",
                    api_key=os.getenv("GITHUB_TOKEN"),
                    base_url="https://models.github.ai"
                )
            
            # Fallback to Azure OpenAI
            elif os.getenv("AZURE_OPENAI_API_KEY"):
                return AzureOpenAIChatClient(
                    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
                    deployment_name=os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME"),
                    endpoint=os.getenv("AZURE_OPENAI_ENDPOINT")
                )
            
            # Try Azure CLI credential as last resort
            else:
                return AzureOpenAIChatClient(
                    credential=DefaultAzureCredential(),
                    deployment_name=os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME", "gpt-4o"),
                    endpoint=os.getenv("AZURE_OPENAI_ENDPOINT")
                )
                
        except Exception as e:
            print(f"⚠️ Error setting up chat client: {e}")
            print("💡 Please set GITHUB_TOKEN or AZURE_OPENAI_API_KEY in your environment")
            raise
    
    def _create_search_function(self):
        """Create the vector search function as a tool."""
        
        def search_documents(
            query: Annotated[str, Field(description="The search query to find semantically relevant documents")]
        ) -> str:
            """Search for documents using vector similarity (semantic search).
            
            This uses embeddings to find documents with similar meaning, 
            not just keyword matches.
            """
            # Perform semantic search using ChromaDB
            results = self.vector_store.query(
                query_texts=[query],
                n_results=3,  # Get top 3 most relevant
                include=["documents", "metadatas", "distances"]
            )
            
            if not results["documents"] or not results["documents"][0]:
                return "No relevant documents found for your query."
            
            # Format the results
            response = "📚 **Semantic Search Results** (using vector similarity):\n\n"
            
            for i, (doc, metadata, distance) in enumerate(zip(
                results["documents"][0],
                results["metadatas"][0],
                results["distances"][0]
            ), 1):
                # Convert distance to similarity score (lower distance = higher similarity)
                similarity = 1 / (1 + distance)  # Simple conversion
                
                response += f"**Document {i}** (Similarity: {similarity:.2%})\n"
                response += f"Title: {metadata['title']}\n"
                response += f"Category: {metadata['category']}\n"
                response += f"Content: {doc}\n\n"
            
            return response
        
        return search_documents
    
    async def ask_question(self, question: str) -> str:
        """Ask a question and get an answer using vector-based RAG."""
        result = await self.agent.run(question)
        return result.text

# Create our Vector RAG agent
rag_agent = VectorRAGAgent(collection)

print("✅ Vector RAG Agent created successfully!")
print("🧮 Using semantic search with vector embeddings")
print("🔍 Can find documents by meaning, not just keywords")
print("🎯 Ready to answer questions!")

## 🧪 Step 3: Test Vector-Based RAG

Let's test our RAG agent with semantic search! Notice how it can find relevant documents even when we don't use exact keywords.

In [ ]:
# 🧪 Test 1: Semantic Search (finds by meaning)
print("🧪 **Test 1: Semantic Search**")
print("=" * 50)

question1 = "What are AI agents and how do they work?"
print(f"❓ Question: {question1}")
print("\n🔍 Searching using vector similarity...")
print("   (This will find documents about AI agents by semantic meaning)\n")

answer1 = await rag_agent.ask_question(question1)
print(f"🤖 Answer:\n{answer1}")
print("=" * 50)

In [ ]:
# 🧪 Test 2: Finding documents without exact keywords
print("\n🧪 **Test 2: Semantic Understanding**")
print("=" * 50)

question2 = "How can I enhance language models with external knowledge?"
print(f"❓ Question: {question2}")
print("\n🔍 Notice: No exact keywords like 'RAG' or 'retrieval' in the question")
print("   But semantic search will still find the RAG document!\n")

answer2 = await rag_agent.ask_question(question2)
print(f"🤖 Answer:\n{answer2}")
print("=" * 50)

In [ ]:
# 🧪 Test 3: Cross-domain semantic search
print("\n🧪 **Test 3: Cross-Domain Search**")
print("=" * 50)

question3 = "What programming language is best for building intelligent systems?"
print(f"❓ Question: {question3}")
print("\n🔍 Semantic search will connect 'intelligent systems' with 'AI development'")
print("   and find Python as the relevant answer!\n")

answer3 = await rag_agent.ask_question(question3)
print(f"🤖 Answer:\n{answer3}")
print("=" * 50)

print("\n✨ **Key Takeaway:**")
print("Vector embeddings enable semantic search - finding documents by meaning,")
print("not just keyword matching. This makes RAG much more powerful!")

## 🎉 Congratulations! You Built a Vector-Based RAG System!

### 🏆 What You Accomplished

You've successfully created a **production-ready RAG system** using:

**🧮 Vector Embeddings:**
- Converted documents into numerical vectors that capture semantic meaning
- Used sentence-transformers for automatic embedding generation
- Stored embeddings in ChromaDB vector database

**🔍 Semantic Search:**
- Implemented similarity search using vector distance metrics
- Found relevant documents by meaning, not just keywords
- Achieved better retrieval than simple text matching

**🤖 AI Agent Integration:**
- Created an agent that uses vector search as a tool
- Agent autonomously decides when to search for information
- Provides comprehensive answers with proper citations

### 🎓 Key RAG Concepts You Learned

**? Vector Embeddings:**
- Transform text into dense numerical representations
- Capture semantic meaning and relationships
- Enable similarity comparisons between texts

**💾 Vector Databases:**
- Store embeddings efficiently for fast retrieval
- Support similarity search at scale
- Handle millions of documents with sub-second queries

**🔗 RAG Pipeline:**
```
User Question 
  → Convert to embedding 
  → Search vector store 
  → Retrieve relevant docs 
  → Generate answer with context
```

### 🚀 Next Steps

**Enhance Your RAG System:**
- ? **Scale Up** - Add thousands of documents to your knowledge base
- 🎯 **Improve Retrieval** - Use hybrid search (keywords + vectors)
- 🧠 **Better Embeddings** - Try different embedding models
- 💬 **Add Memory** - Enable multi-turn conversations
- ? **Iterative Retrieval** - Search multiple times for complex questions

**Production Considerations:**
- 🗄️ **Persistent Storage** - Use ChromaDB with persistent storage
- ⚡ **Performance** - Optimize embedding generation and search
- 📊 **Monitoring** - Track retrieval quality and answer accuracy
- 🔒 **Security** - Implement access control and data filtering

### 🛠️ Tools & Technologies Used

- **Agent Framework** - Microsoft's agent orchestration framework
- **ChromaDB** - Lightweight vector database
- **Sentence Transformers** - Pre-trained embedding models  
- **OpenAI/Azure OpenAI** - Language model for generation

### 💡 Real-World Applications

Your RAG skills can power:
- **📚 Document Q&A** - Chat with PDFs, docs, knowledge bases
- **🔧 Customer Support** - Automated help with company knowledge
- **📊 Research Assistants** - Find and synthesize information
- **💼 Enterprise Search** - Semantic search across company data
- **🎓 Educational Tools** - Personalized learning assistants

---

**🎊 You now understand how to build RAG systems with vector embeddings and semantic search!**

**Ready to build more complex agents? Check out the next notebooks in this series!** 🚀